# Module 05 — Loss Functions, Optimizers & the Training Loop

**Prerequisites:** Module 04 (Building Neural Networks: From Scratch to `nn.Module`)
**Time:** ~120 minutes

## Learning Objectives

- Choose the correct loss function for a regression vs. a classification task.
- Explain what `torch.optim` optimizers do, and how they use `.grad`.
- Write the canonical 5-step PyTorch training loop from memory, and explain every line.
- Add a validation loop and understand `model.train()` vs `model.eval()`.
- Plot training curves and read them for signs of learning, overfitting, or divergence.
- Compare SGD and Adam optimizers and choose appropriate learning rates.
- Implement basic learning rate scheduling and early stopping.
- Diagnose the most common "my model isn't learning" bugs related to this loop.


In [ ]:
import torch
import torch.nn as nn


## Loss Functions: Measuring "How Wrong"

A loss function takes the model's prediction and the true target, and returns a single number: how bad the prediction is. Lower is better; a perfect prediction gives a loss of 0.

| Loss | Task | What it expects |
|---|---|---|
| `nn.MSELoss()` | Regression (predicting a continuous number) | Predictions and targets, any matching shape |
| `nn.CrossEntropyLoss()` | Multi-class classification | **Raw logits** (not softmax!) and integer class labels |
| `nn.BCEWithLogitsLoss()` | Binary classification | **Raw logits** (not sigmoid!) and 0/1 float targets |

### Mean Squared Error (Regression)

MSE computes the average of `(prediction - target)^2` across all examples. Squaring makes all errors positive and penalizes large errors more heavily than small ones.


In [ ]:
predictions = torch.tensor([2.5, 0.0, 2.1])
targets     = torch.tensor([3.0, -0.5, 2.0])

mse = nn.MSELoss()
loss = mse(predictions, targets)
print("MSE loss:", loss.item())

# Manually verifying the formula: mean of (pred - target)^2
manual = ((predictions - targets) ** 2).mean()
print("manual   :", manual.item())

# L1Loss / MAE — alternative for regression, less sensitive to outliers
mae = nn.L1Loss()
print("MAE loss:", mae(predictions, targets).item())


### Cross-Entropy (Multi-class Classification)

This is the one that trips up nearly every beginner, so let's be explicit: `nn.CrossEntropyLoss` expects **raw, unnormalized scores** from the model (called "logits") — *not* probabilities you've already run through softmax. It applies `log_softmax` and negative-log-likelihood *internally*, in one numerically stable step.


In [ ]:
# 3 samples, 4 possible classes each -- these are RAW LOGITS, not probabilities
logits = torch.tensor([
    [2.0, 1.0, 0.1, 0.0],   # model is fairly confident about class 0
    [0.1, 0.2, 3.0, 0.1],   # model is fairly confident about class 2
    [1.0, 1.0, 1.0, 1.0],   # model has no idea (all equal)
])
true_classes = torch.tensor([0, 2, 1])   # the correct class index for each sample -- NOT one-hot!

ce = nn.CrossEntropyLoss()
loss = ce(logits, true_classes)
print("cross-entropy loss:", loss.item())


Two details worth memorizing, because getting either wrong produces a model that trains "successfully" but learns the wrong thing:

1. **Don't apply softmax yourself** before passing predictions into `nn.CrossEntropyLoss`. If you do, you're applying softmax *twice* (once manually, once inside the loss function), which silently produces incorrect gradients.
2. **Targets are integer class indices** (e.g., `2` meaning "class 2"), not one-hot vectors (`[0,0,1,0]`).

If you need actual probabilities for reporting/inspection (not for computing loss), apply `torch.softmax(logits, dim=-1)` separately, purely for display.


In [ ]:
probabilities = torch.softmax(logits, dim=-1)
print(probabilities)
print("each row sums to 1.0:", probabilities.sum(dim=-1))

# Getting predicted classes from logits
predicted_classes = logits.argmax(dim=-1)
print("\npredicted:", predicted_classes)
print("true:     ", true_classes)
accuracy = (predicted_classes == true_classes).float().mean()
print(f"accuracy: {accuracy.item():.1%}")


### Binary Cross-Entropy (Binary Classification)

For a yes/no, 0/1 classification task, `nn.BCEWithLogitsLoss` is the standard choice — again, it expects raw logits (a single number per sample, positive meaning "leans toward class 1"), and applies sigmoid internally.


In [ ]:
logits_binary = torch.tensor([2.0, -1.0, 0.1])   # raw scores, one per sample
targets_binary = torch.tensor([1.0, 0.0, 1.0])     # 0 or 1, as floats

bce = nn.BCEWithLogitsLoss()
loss = bce(logits_binary, targets_binary)
print("binary cross-entropy loss:", loss.item())

# Converting logits to predicted classes:
predicted = (logits_binary > 0).float()   # logit > 0 <=> sigmoid(logit) > 0.5
print("predicted:", predicted)
print("accuracy:", (predicted == targets_binary).float().mean().item())


## Optimizers: Automating the Update Step

Recall Module 04's manual update:

```python
with torch.no_grad():
    W -= learning_rate * W.grad
    b -= learning_rate * b.grad
W.grad.zero_()
b.grad.zero_()
```

An **optimizer** wraps exactly this logic (and, for more advanced optimizers, quite a bit more bookkeeping) behind two method calls: `optimizer.step()` (apply the update) and `optimizer.zero_grad()` (reset gradients).


In [ ]:
model = nn.Linear(1, 1)

# SGD = Stochastic Gradient Descent -- the update we wrote by hand in Module 04
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

print("Optimizer is tracking these parameter shapes:")
for p in model.parameters():
    print(" ", tuple(p.shape))


### Optimizer Comparison

| Optimizer | Typical use | Typical starting LR | Key property |
|---|---|---|---|
| `SGD(params, lr, momentum=...)` | Simple baselines, well-tuned CNN pipelines | 0.01–0.1 | Simple, requires more tuning |
| `Adam(params, lr)` | Good general-purpose default | 1e-3 | Adaptive per-parameter LR |
| `AdamW(params, lr, weight_decay=...)` | Fine-tuning, transformer-style models | 1e-4–1e-3 | Correct weight decay |

`Adam` is the most common starting point in practice — it adapts its effective step size per-parameter, which usually makes it more forgiving of an imperfectly-chosen learning rate than plain `SGD`.


## The Canonical Training Loop

Here is the pattern used in essentially every PyTorch project you will ever encounter. Memorize the five steps inside the inner loop; they don't change, regardless of model architecture or task.


In [ ]:
torch.manual_seed(0)

# Synthetic regression data
x = torch.linspace(-5, 5, 200).unsqueeze(1)
y = 3 * x + 2 + torch.randn_like(x) * 0.5

model = nn.Linear(1, 1)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

n_epochs = 100
for epoch in range(n_epochs):
    # 1. Reset gradients from the previous iteration (Module 03: they accumulate otherwise)
    optimizer.zero_grad()

    # 2. Forward pass: compute predictions
    predictions = model(x)

    # 3. Compute the loss: how wrong are the predictions?
    loss = loss_fn(predictions, y)

    # 4. Backward pass: compute d(loss)/d(every parameter), via autograd
    loss.backward()

    # 5. Update parameters using the computed gradients
    optimizer.step()

    if epoch % 20 == 0:
        print(f"epoch {epoch:3d} | loss {loss.item():.4f}")

print(f"\nlearned weight: {model.weight.item():.3f} (true=3.0)")
print(f"learned bias:   {model.bias.item():.3f} (true=2.0)")


```text
Data
 |
 v
Model            <- forward pass (step 2)
 |
 v
Prediction
 |
 v
Loss             <- compare to target (step 3)
 |
 v
Backward pass    <- loss.backward() (step 4)
 |
 v
Gradients        <- stored in param.grad
 |
 v
Optimizer        <- optimizer.step() (step 5)
 |
 v
Updated parameters
 |
 v
Repeat
```


## `model.train()` vs `model.eval()`

Some layers (dropout, batch normalization — introduced later) behave *differently* during training vs evaluation. These two methods switch the model between modes:

- **`model.train()`** — activates training behavior (dropout drops, batchnorm uses batch stats)
- **`model.eval()`** — activates evaluation behavior (dropout disabled, batchnorm uses running stats)

Always use them, even if your current model doesn't have those layers — it costs nothing and prevents bugs later.


In [ ]:
print("training mode:", model.training)   # True by default

model.eval()
print("after model.eval():", model.training)   # False

model.train()
print("after model.train():", model.training)   # True again


## Adding a Validation Loop

The training loop tells you how well the model fits the *training* data. To know how well it *generalizes* to unseen data, you need a **validation loop** that evaluates on held-out data without updating the model.


In [ ]:
torch.manual_seed(0)

# Split data into train (80%) and validation (20%)
n = len(x)
n_train = int(0.8 * n)
x_train, x_val = x[:n_train], x[n_train:]
y_train, y_val = y[:n_train], y[n_train:]

model = nn.Linear(1, 1)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

train_losses = []
val_losses = []

for epoch in range(100):
    # --- Training ---
    model.train()
    optimizer.zero_grad()
    pred_train = model(x_train)
    loss_train = loss_fn(pred_train, y_train)
    loss_train.backward()
    optimizer.step()
    train_losses.append(loss_train.item())
    
    # --- Validation ---
    model.eval()
    with torch.no_grad():   # no gradient tracking during evaluation
        pred_val = model(x_val)
        loss_val = loss_fn(pred_val, y_val)
    val_losses.append(loss_val.item())
    
    if epoch % 20 == 0:
        print(f"epoch {epoch:3d} | train_loss {loss_train.item():.4f} | val_loss {loss_val.item():.4f}")

print(f"\nFinal train loss: {train_losses[-1]:.4f}")
print(f"Final val loss:   {val_losses[-1]:.4f}")


## Reading Training Curves

Tracking train loss and val loss across epochs tells you everything about how training is going:

| Pattern | What it means |
|---|---|
| Both decrease together | ✅ Healthy learning |
| Train ↓, Val ↓ then ↑ | ⚠️ Overfitting (model memorizes training data) |
| Both plateau high | ⚠️ Underfitting (model too simple or LR too small) |
| Loss increases/oscillates | ❌ Learning rate too high |
| Loss becomes NaN | ❌ Numerical instability (LR too high, or bad data) |


In [ ]:
# Text-based loss curve (works without matplotlib)
def print_loss_curve(train_losses, val_losses, width=60):
    max_loss = max(max(train_losses[:20]), max(val_losses[:20]))
    print(f"\n{'Epoch':>5} | {'Train':>8} | {'Val':>8} | Loss Curve")
    print("-" * (width + 30))
    
    # Show every 5th epoch
    for i in range(0, len(train_losses), 5):
        t = train_losses[i]
        v = val_losses[i]
        t_bar = int(t / max_loss * width / 2)
        v_bar = int(v / max_loss * width / 2)
        print(f"{i:5d} | {t:8.4f} | {v:8.4f} | {'█' * t_bar}{'░' * v_bar}")

print_loss_curve(train_losses, val_losses)


## Adam vs SGD: A Comparison

Let's train the same model with both optimizers and compare convergence speed.


In [ ]:
def train_and_track(optimizer_class, lr, n_epochs=100):
    torch.manual_seed(42)
    model = nn.Linear(1, 1)
    loss_fn = nn.MSELoss()
    optimizer = optimizer_class(model.parameters(), lr=lr)
    losses = []
    
    for epoch in range(n_epochs):
        optimizer.zero_grad()
        loss = loss_fn(model(x_train), y_train)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    
    return losses

sgd_losses = train_and_track(torch.optim.SGD, lr=0.01)
adam_losses = train_and_track(torch.optim.Adam, lr=0.1)

print(f"{'Epoch':>5} | {'SGD (lr=0.01)':>14} | {'Adam (lr=0.1)':>14}")
print("-" * 40)
for epoch in [0, 5, 10, 20, 50, 99]:
    print(f"{epoch:5d} | {sgd_losses[epoch]:14.4f} | {adam_losses[epoch]:14.4f}")


## Multi-class Classification: End-to-End

Let's build a complete multi-class classification pipeline — generating synthetic data, training, and evaluating.


In [ ]:
torch.manual_seed(0)

# Generate 3-class data: 300 samples, 2 features
n_per_class = 100
features = torch.cat([
    torch.randn(n_per_class, 2) + torch.tensor([2.0, 2.0]),   # class 0
    torch.randn(n_per_class, 2) + torch.tensor([-2.0, 2.0]),  # class 1
    torch.randn(n_per_class, 2) + torch.tensor([0.0, -2.0]),  # class 2
])
labels = torch.cat([torch.zeros(n_per_class), torch.ones(n_per_class), torch.full((n_per_class,), 2)]).long()

# Shuffle
perm = torch.randperm(len(labels))
features, labels = features[perm], labels[perm]

# Split
n_train = 240
X_train, X_val = features[:n_train], features[n_train:]
y_train, y_val = labels[:n_train], labels[n_train:]

print(f"Train: {len(X_train)} samples, Val: {len(X_val)} samples")
print(f"Class distribution (train): {torch.bincount(y_train)}")


In [ ]:
# Model, loss, optimizer
model = nn.Sequential(
    nn.Linear(2, 16),
    nn.ReLU(),
    nn.Linear(16, 3),   # 3 output classes (raw logits)
)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Training with validation
for epoch in range(100):
    model.train()
    optimizer.zero_grad()
    logits = model(X_train)
    loss = loss_fn(logits, y_train)
    loss.backward()
    optimizer.step()
    
    if epoch % 20 == 0 or epoch == 99:
        model.eval()
        with torch.no_grad():
            val_logits = model(X_val)
            val_loss = loss_fn(val_logits, y_val)
            val_preds = val_logits.argmax(dim=1)
            val_acc = (val_preds == y_val).float().mean()
        
        train_preds = logits.argmax(dim=1)
        train_acc = (train_preds == y_train).float().mean()
        
        print(f"epoch {epoch:3d} | loss {loss.item():.4f} | train_acc {train_acc:.3f} | val_acc {val_acc:.3f}")


## Learning Rate Scheduling (Basics)

Often, a fixed learning rate isn't optimal throughout training. **LR schedulers** adjust the learning rate as training progresses.


In [ ]:
model = nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

# StepLR: reduce LR by a factor every N epochs
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.1)

print("LR schedule (StepLR, step=30, gamma=0.1):")
for epoch in range(100):
    if epoch % 10 == 0:
        print(f"  epoch {epoch:3d}: lr = {scheduler.get_last_lr()[0]:.6f}")
    # ... training step would go here ...
    scheduler.step()   # must be called AFTER optimizer.step()


In [ ]:
# CosineAnnealingLR: smoothly decrease LR from initial to near-zero
model = nn.Linear(1, 1)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)

print("LR schedule (CosineAnnealing, T_max=100):")
for epoch in range(100):
    if epoch % 10 == 0:
        print(f"  epoch {epoch:3d}: lr = {scheduler.get_last_lr()[0]:.6f}")
    scheduler.step()


## Early Stopping

If the validation loss hasn't improved for `patience` epochs, stop training — continuing would only make overfitting worse.


In [ ]:
class EarlyStopping:
    def __init__(self, patience=10, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float('inf')
        self.counter = 0
        self.should_stop = False
    
    def __call__(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
        return self.should_stop

# Usage in a training loop:
early_stop = EarlyStopping(patience=5)
# Simulate losses
fake_val_losses = [1.0, 0.8, 0.6, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5]
for epoch, val_loss in enumerate(fake_val_losses):
    if early_stop(val_loss):
        print(f"Early stopping at epoch {epoch}!")
        break
    print(f"epoch {epoch}: val_loss={val_loss:.4f}, patience_counter={early_stop.counter}")


## Gradient Clipping in the Training Loop

When training deep networks, gradient clipping prevents exploding gradients. Add it between `loss.backward()` and `optimizer.step()`.


In [ ]:
model = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 3))
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Inside the training loop:
optimizer.zero_grad()
logits = model(X_train)
loss = loss_fn(logits, y_train)
loss.backward()

# Clip gradients AFTER backward, BEFORE step
grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
print(f"gradient norm (before clipping): {grad_norm:.4f}")

optimizer.step()


## 🐛 Debugging Challenge: Forgetting `zero_grad()`

Predict what happens to the loss curve if we drop `optimizer.zero_grad()` from the loop.


In [ ]:
torch.manual_seed(0)
model_bug = nn.Linear(1, 1)
optimizer_bug = torch.optim.SGD(model_bug.parameters(), lr=0.01)

for epoch in range(100):
    # optimizer_bug.zero_grad()   <-- MISSING ON PURPOSE
    predictions = model_bug(x_train)
    loss = loss_fn(predictions, y_train)
    loss.backward()
    optimizer_bug.step()
    if epoch % 20 == 0:
        print(f"epoch {epoch:3d} | loss {loss.item():.4f}")


Without `zero_grad()`, the loss decreases far more slowly, plateaus at a worse value, or behaves erratically. This bug **does not crash** — it just quietly makes training worse, which is exactly what makes it dangerous.


## 🐛 Debugging Challenge: Learning Rate Too High


In [ ]:
torch.manual_seed(0)
model_diverge = nn.Linear(1, 1)
optimizer_diverge = torch.optim.SGD(model_diverge.parameters(), lr=5.0)   # WAY too high

for epoch in range(10):
    optimizer_diverge.zero_grad()
    predictions = model_diverge(x_train)
    loss = loss_fn(predictions, y_train)
    loss.backward()
    optimizer_diverge.step()
    print(f"epoch {epoch} | loss {loss.item():.4f}")


If the loss is growing (or becomes `nan`/`inf`), the learning rate is too high. Each update overshoots past the minimum. The fix: try a smaller learning rate (often by 10x).


## Metric Tracking Pattern

A clean pattern for tracking all metrics during training:


In [ ]:
# Clean metric tracking with a dictionary
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

torch.manual_seed(0)
model = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 3))
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(50):
    # Train
    model.train()
    optimizer.zero_grad()
    logits = model(X_train)
    loss = loss_fn(logits, y_train)
    loss.backward()
    optimizer.step()
    
    train_acc = (logits.argmax(1) == y_train).float().mean().item()
    history['train_loss'].append(loss.item())
    history['train_acc'].append(train_acc)
    
    # Validate
    model.eval()
    with torch.no_grad():
        val_logits = model(X_val)
        val_loss = loss_fn(val_logits, y_val)
        val_acc = (val_logits.argmax(1) == y_val).float().mean().item()
    history['val_loss'].append(val_loss.item())
    history['val_acc'].append(val_acc)

# Print final summary
print(f"\n{'Metric':>15} | {'Start':>8} | {'End':>8} | {'Best':>8}")
print("-" * 50)
for key, values in history.items():
    best = min(values) if 'loss' in key else max(values)
    print(f"{key:>15} | {values[0]:8.4f} | {values[-1]:8.4f} | {best:8.4f}")


## Exercises

🟢 **Beginner 1:** Using the `logits` and `true_classes` tensors from earlier, compute the predicted class for each sample using `logits.argmax(dim=-1)`, and compare against `true_classes` to compute accuracy.

🟢 **Beginner 2:** Write the 5-step training loop from memory (without looking above). Use `nn.Linear(1,1)`, `nn.MSELoss()`, and `torch.optim.SGD` with `lr=0.01`.

🟡 **Intermediate 1:** Repeat the training loop using `torch.optim.Adam(model.parameters(), lr=0.01)` instead of SGD. Compare how many epochs it takes to reach a loss below 0.3.

🟡 **Intermediate 2:** Add a `CosineAnnealingLR` scheduler to the classification training loop. Print the learning rate at each epoch and observe how it changes.

🔴 **Challenge 1:** Build a full binary classification training loop. Generate data: `x = torch.randn(200, 2)`, `y = (x[:, 0] + x[:, 1] > 0).float()`. Use `nn.BCEWithLogitsLoss()`, `Adam`, and track train/val accuracy.

🔴 **Challenge 2:** Implement the `EarlyStopping` class in a real training loop. Train until early stopping triggers or 200 epochs pass. Print which epoch stopped training.


In [ ]:
# Space for your exercise solutions



## Common Mistakes

- **Applying softmax/sigmoid before `nn.CrossEntropyLoss`/`nn.BCEWithLogitsLoss`** — both expect raw logits; doing it twice silently breaks gradients.
- **Forgetting `optimizer.zero_grad()`** — trains, but poorly, with no error to signal the problem.
- **Forgetting `model.eval()` and `torch.no_grad()` during validation** — wastes memory and gives wrong results once dropout/batchnorm are added.
- **Learning rate too high** — loss grows or oscillates instead of shrinking; reduce by 10x.
- **Learning rate too low** — loss shrinks agonizingly slowly; try increasing it.
- **Wrong target format** — `CrossEntropyLoss` wants integer class indices, `BCEWithLogitsLoss` wants float 0/1.
- **Calling `scheduler.step()` before `optimizer.step()`** — in most cases, scheduler should be called after the optimizer.

## Mental Model

The training loop is a feedback cycle: *guess* (forward) → *grade* (loss) → *improve* (backward + step) → *forget the old feedback* (zero_grad) → *repeat*. Every training loop, from 2-parameter regression to billion-parameter LLMs, is this same cycle.

## Key Takeaways

- Choose your loss function based on the task: `MSELoss` for regression, `CrossEntropyLoss` for multi-class, `BCEWithLogitsLoss` for binary — always pass raw logits to classification losses.
- The 5-step training loop (zero_grad → forward → loss → backward → step) is universal.
- Always use `model.train()`/`model.eval()` and `torch.no_grad()` appropriately.
- Track both train and validation metrics to detect overfitting.
- Adam is a good default optimizer; LR schedulers and early stopping improve generalization.
- A loss that doesn't decrease usually traces back to: missing `zero_grad()`, bad LR, or wrong loss/target format.

## What's Next

**Module 06 — Datasets & DataLoaders** replaces the toy in-memory tensors with PyTorch's `Dataset` and `DataLoader` abstractions for proper batch-by-batch training.

## Checklist

- [ ] I can choose the correct loss function for regression, multi-class, and binary classification
- [ ] I understand why classification losses expect raw logits, not softmax/sigmoid output
- [ ] I can write the 5-step training loop from memory and explain each line
- [ ] I can add a validation loop with `model.eval()` and `torch.no_grad()`
- [ ] I can track and interpret training curves (loss + accuracy)
- [ ] I can recognize the symptoms of missing `zero_grad()` and bad learning rates
- [ ] I know how to use basic LR scheduling and early stopping
